# Capybara RC3 — Sphinx recursion hotfix

Run the cell below from the root of the existing local Git clone. It repairs the recursive mirrored-document include, patches the RC3 finalizer, commits, and pushes `main`.


In [ ]:
# CAPYBARA RC3 — HOTFIX: MyST/Sphinx recursive mirrored-document include
#
# Run this from the ROOT of your local Git clone:
#   BrianBowers-NapaCounty/itam-itsm_integration_framework
#
# This:
#   1) restores CONTRIBUTING.md as a real canonical document,
#   2) materializes docs/contributing.md, docs/changelog.md, and
#      docs/code-of-conduct.md as direct content instead of recursive includes,
#   3) patches tools/finalize_rc3.py so rerunning finalization cannot recreate
#      the recursion,
#   4) removes any stale generated *.md.rst wrappers if they exist,
#   5) commits and pushes the hotfix to origin/main.
#
# Do NOT "fix" this by raising sys.setrecursionlimit(). The recursion is a
# source-structure bug, not a legitimately deep document.

from pathlib import Path
import subprocess, re, sys, importlib.util

ROOT = Path.cwd().resolve()

required = [
    ROOT / ".git",
    ROOT / ".readthedocs.yaml",
    ROOT / "docs",
    ROOT / "tools" / "finalize_rc3.py",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise RuntimeError(
        "Run this cell from the LOCAL GIT REPOSITORY ROOT.\nMissing:\n  "
        + "\n  ".join(missing)
    )

def git(*args, check=True):
    r = subprocess.run(
        ["git", "-C", str(ROOT), *args],
        capture_output=True,
        text=True
    )
    if r.stdout.strip():
        print(r.stdout.strip())
    if r.stderr.strip():
        print(r.stderr.strip())
    if check and r.returncode != 0:
        raise RuntimeError(
            f"git {' '.join(args)} failed ({r.returncode})\n{r.stderr}"
        )
    return r

print("Repository:", ROOT)
print("Branch:", git("branch", "--show-current").stdout.strip())

CONTRIBUTING = r'''# Contributing to Capybara

Thank you for your interest in contributing to Capybara. Community contributions are welcome, encouraged, and essential to the long-term health of this project.

This document explains how to contribute in a way that is consistent with the project's goals, structure, and licensing.

---

## License Context

This repository is licensed under the MIT License, as described in the `LICENSE` file.

By submitting a contribution, you agree that:

- Your contributions are provided freely and openly to the global community.
- Your contributions may be used, modified, merged, published, distributed, sublicensed, and/or sold, consistent with the MIT License.
- You grant the project maintainers and all downstream users the same permissions granted by the MIT License, without additional restrictions.

### Attribution Requirement

While the MIT License permits broad reuse, this project includes the following project-level requirement, which contributors must respect:

- The file `AUTHORS.md` must not be modified.
- The file `AUTHORS.md` must be included in all future forks, redistributions, and derivative works of this repository.

This requirement exists to preserve authorship history and credit. Contributions that attempt to remove, alter, or bypass this requirement will not be accepted.

---

## How to Contribute

You may contribute in many ways, including but not limited to:

- Documentation improvements
- New content, examples, or frameworks
- Bug fixes or corrections
- Structural or organizational improvements
- Clarifications or refinements to existing material

All contributions should be made via pull requests and should follow the conventions described below.

---

## Repository Conventions

To keep the repository organized, maintainable, and navigable, contributors are expected to follow these conventions.

### Folder Structure

- Whenever possible, new content should be added to an appropriate subfolder, whether existing or newly created.
- Adding content directly to the top-level `Capybara` folder is strongly discouraged, except where explicitly required by existing structure or maintainers.
- If a suitable subfolder does not exist, contributors are encouraged to create one.

### README Files

- All new subfolders must include a `README.md` describing their purpose, contents, and intended usage.
- For existing subfolders, contributors should update and maintain the associated `README.md` when changes are made.

Clear, accurate documentation is considered a core part of any contribution.

### Changelog Updates

- The file `CHANGELOG.md` in the top-level `Capybara` folder must always be updated to reflect any additions, removals, or modifications made to the repository.
- Changelog entries should be concise, descriptive, and grouped under the appropriate version heading.

Pull requests that modify content without a corresponding changelog update may be requested to revise before acceptance.

---

## Versioning and Releases

This project follows Semantic Versioning (SemVer):

`MAJOR.MINOR.PATCH`

Each version must be preceded by an Edition Codename, chosen by the contributor or release author.

Examples:

- `Capybara: Purple Edition 3.5.14`
- `Capybara: Verdant Edition 2.1.0`

### Initial Release

The initial public release, planned for September 2026, will be:

- **Capybara: Original Edition 1.0.0**

Subsequent releases should increment version numbers according to SemVer rules and include meaningful changelog entries.

---

## Publishing a New Edition and Version

Capybara editions and versions are published using standard Git and GitHub workflows, following Semantic Versioning (SemVer) and the project's Edition + Version naming convention.

The **Publishing a New Edition** section explains how to publish a new edition using either the command-line Git workflow or the GitHub web interface, using the transition from:

> Capybara: Alpha Edition 0.x.y to Capybara: Original Edition 1.0.0

as a real-world example.

### After Publishing

Once a new edition is published:

- The release tag (`v1.0.0`) becomes the canonical reference point for that edition.
- Documentation hosting platforms (such as Read the Docs) may be configured to expose the version publicly.
- All future development should proceed on `main` toward the next edition or version (for example, `1.1.0` or a future edition).

### Notes on Pre-Releases

Pre-release editions (such as Alpha or Beta editions using `0.x.y` versions):

- May be published without tags.
- Should clearly indicate their pre-release status in documentation.
- Do not imply long-term stability or backward compatibility.

The Original Edition 1.0.0 represents Capybara's first stable, production-ready release and establishes the baseline for all future versions.

By following these steps, contributors help ensure Capybara releases remain consistent, traceable, and easy for the global community to adopt and build upon.

---

## Style Notes and Guidelines

The **Style Notes and Guidelines** section in the project's guide back-matter establishes the baseline stylistic and structural conventions for this repository.

- All contributors are expected to honor the examples and guidelines in that section.
- Deviations from established style patterns are discouraged, but not forbidden.

### Intentional Style Changes

If a contributor makes a carefully considered decision to depart from existing style precedents:

- The new version's **Style Notes and Guidelines** section must be updated to document and justify the new style choices.
- The change should be explained clearly in the changelog.

This ensures that style evolution is intentional, documented, and understandable to future contributors.

---

## Original Edition 1.0.0 Contribution Requirements

Contributions should preserve authoritative-system boundaries, platform-neutral core logic, U.S. English spelling, accessible documentation, and dry-run-first examples.

Open an issue or pull request describing the use case, affected platforms, data ownership, and test approach.

---

## Final Notes

- Contributions should be respectful, constructive, and well-documented.
- Large or structural changes are encouraged to be discussed in an issue before submission.
- Maintainers reserve the right to request revisions to ensure consistency with this guide.

Thank you for helping make Capybara better for everyone.

---
'''

(ROOT / "CONTRIBUTING.md").write_text(CONTRIBUTING, encoding="utf-8")

def write_mirror(root_name, docs_name):
    src = ROOT / root_name
    dst = ROOT / "docs" / docs_name
    if not src.exists():
        raise RuntimeError(f"Canonical root file missing: {src}")
    canonical = src.read_text(encoding="utf-8", errors="replace").strip()
    note = (
        f"This page mirrors the repository's `/{root_name}`. "
        "The documentation copy is materialized during release finalization "
        "to avoid recursive include behavior in MyST/Sphinx.\n\n"
    )
    dst.write_text(note + canonical + "\n", encoding="utf-8")
    print("Materialized:", dst.relative_to(ROOT))

write_mirror("CONTRIBUTING.md", "contributing.md")
write_mirror("CHANGELOG.md", "changelog.md")
write_mirror("CODE_OF_CONDUCT.md", "code-of-conduct.md")

stale = [
    ROOT / "CONTRIBUTING.md.rst",
    ROOT / "docs" / "contributing.md.rst",
    ROOT / "docs" / "changelog.md.rst",
    ROOT / "docs" / "code-of-conduct.md.rst",
]
for p in stale:
    if p.exists():
        p.unlink()
        print("Removed stale wrapper:", p.relative_to(ROOT))

finalizer = ROOT / "tools" / "finalize_rc3.py"
s = finalizer.read_text(encoding="utf-8")

dangerous = '''    if (docs / "contributing.md").exists():
        shutil.copy2(docs / "contributing.md", repo / "CONTRIBUTING.md")
'''

replacement = '''    # Canonical mirrored repository documents flow ROOT -> docs only.
    # Never copy docs/contributing.md back over root CONTRIBUTING.md:
    # the historical docs page may contain a MyST include of the root file,
    # which would make the root file recursively include itself.
    for root_name, docs_name in [
        ("CONTRIBUTING.md", "contributing.md"),
        ("CHANGELOG.md", "changelog.md"),
        ("CODE_OF_CONDUCT.md", "code-of-conduct.md"),
    ]:
        source = repo / root_name
        destination = docs / docs_name
        if source.exists():
            canonical = source.read_text(encoding="utf-8", errors="replace").strip()
            note = (
                f"This page mirrors the repository's `/{root_name}`. "
                "The documentation copy is materialized during release finalization "
                "to avoid recursive include behavior in MyST/Sphinx.\\n\\n"
            )
            destination.write_text(note + canonical + "\\n", encoding="utf-8")
'''

if dangerous in s:
    s = s.replace(dangerous, replacement)
elif "Never copy docs/contributing.md back over root CONTRIBUTING.md" not in s:
    raise RuntimeError(
        "Could not identify the expected RC3 finalizer block. "
        "Stop here rather than patching an unknown revision."
    )

finalizer.write_text(s, encoding="utf-8")
print("Patched:", finalizer.relative_to(ROOT))

bad = []
for rel in [
    "CONTRIBUTING.md",
    "docs/contributing.md",
    "docs/changelog.md",
    "docs/code-of-conduct.md",
]:
    t = (ROOT / rel).read_text(encoding="utf-8", errors="replace")
    if "```{include}" in t or "{include}" in t or ".. include::" in t:
        bad.append(rel)

if bad:
    raise RuntimeError(
        "Recursive/include wrappers still present in: " + ", ".join(bad)
    )

compile(finalizer.read_text(encoding="utf-8"), str(finalizer), "exec")

print("\n✓ Mirrored-document recursion removed.")
print("✓ Root CONTRIBUTING.md restored as direct content.")
print("✓ RC3 finalizer patched against recurrence.")

if importlib.util.find_spec("sphinx") is not None:
    print("\nRunning local Sphinx HTML smoke test...")
    build_dir = ROOT / "_build_hotfix_html"
    r = subprocess.run(
        [
            sys.executable, "-m", "sphinx",
            "-T", "-b", "html",
            str(ROOT / "docs"),
            str(build_dir),
        ],
        text=True
    )
    if r.returncode != 0:
        raise RuntimeError(
            "Local Sphinx smoke test failed. Do NOT push yet; inspect output above."
        )
    print("✓ Local Sphinx HTML build completed.")
else:
    print("\nSphinx is not installed in this kernel; skipping local smoke test.")

git("add", "-A")

status = git("status", "--porcelain", check=False).stdout.strip()
if not status:
    print("\nNo Git changes remain to commit.")
else:
    git(
        "commit",
        "-m",
        "Fix recursive MyST mirror includes in RC3 documentation",
    )

confirm = input(
    "\nType PUSH HOTFIX to push this repair to origin/main: "
).strip()

if confirm != "PUSH HOTFIX":
    raise RuntimeError("Push cancelled.")

git("push", "origin", "main")

print("\n" + "=" * 72)
print("RC3 SPHINX RECURSION HOTFIX PUSHED")
print("=" * 72)
print("Read the Docs should rebuild from the new main commit via its webhook.")
print("If it does not start automatically, trigger a new 'latest' build in RTD.")
print("=" * 72)
